### Food Delivery Time Prediction Project
### Phase 1 - Import Libraries & Load Dataset

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML Tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Load Dataset
df = pd.read_csv("Food_Delivery_Times.csv")

# Show Data
print("Shape of Dataset:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nFirst 5 Rows:")
display(df.head())
df.info()
df.isnull().sum()

### Phase 2 - Data Cleaning + Exploratory Data Analysis
### Remove Useless Column

In [ ]:
df.drop("Order_ID", axis=1, inplace=True)

print("Columns after dropping Order_ID:")
print(df.columns)


In [ ]:
df.isnull().sum()

In [ ]:
# Fill categorical missing values
df["Weather"].fillna(df["Weather"].mode()[0], inplace=True)
df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0], inplace=True)
df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0], inplace=True)
df["Vehicle_Type"].fillna(df["Vehicle_Type"].mode()[0], inplace=True)

# Fill numerical missing values
df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].median(), inplace=True)

print("Missing values after filling:")
print(df.isnull().sum())

In [ ]:
df.describe()


In [ ]:
df.dtypes

### Correlation Heatmap


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

### Scatter Plot

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x="Distance_km", y="Delivery_Time_min", data=df)
plt.title("Distance vs Delivery Time")
plt.show()

### Weather Impact

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Weather", y="Delivery_Time_min", data=df)
plt.title("Weather vs Delivery Time")
plt.show()

### Traffic Impact

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Traffic_Level", y="Delivery_Time_min", data=df)
plt.title("Traffic Level vs Delivery Time")
plt.show()

 ### Vehicle Comparison

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Vehicle_Type", y="Delivery_Time_min", data=df)
plt.title("Vehicle Type vs Delivery Time")
plt.show()

### Time_of_Day

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Time_of_Day", y="Delivery_Time_min", data=df)
plt.title("Time of Day vs Delivery Time")
plt.show()

### # Phase 3 - Data Preprocessing for Machine Learning

### STEP 1 — Define Features and Target

In [ ]:
# Features (X) and Target (y)
X = df.drop("Delivery_Time_min", axis=1)
y = df["Delivery_Time_min"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

display(X.head())
print(y.head())

### STEP 2 — Define Column Types

In [ ]:
# Categorical Columns
cat_cols = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

# Numerical Columns
num_cols = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs"]

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

### STEP 3 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Train Size:", X_train.shape)
print("Test Size:", X_test.shape)

### STEP 4 — Create Best Preprocessor

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", MinMaxScaler(), num_cols)
    ]
)

### STEP 5 — Test Transform

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)

 ### STEP 6 — Understand OneHot Output

In [ ]:
encoded_names = preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols)

print("Encoded Features:")
print(encoded_names)

### Phase 4 - Compare Multiple Regression Models


### STEP 1 — Import Models

In [ ]:
from sklearn.pipeline import Pipeline

# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Tree Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

# Other Models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

### STEP 2 — Add XGBoost / CatBoost if Installed

In [ ]:
models = {}

# Basic Models
models["Linear Regression"] = LinearRegression()
models["Ridge"] = Ridge()
models["Lasso"] = Lasso()
models["ElasticNet"] = ElasticNet()

models["Decision Tree"] = DecisionTreeRegressor(random_state=42)
models["Random Forest"] = RandomForestRegressor(random_state=42)
models["Gradient Boosting"] = GradientBoostingRegressor(random_state=42)
models["Extra Trees"] = ExtraTreesRegressor(random_state=42)
models["AdaBoost"] = AdaBoostRegressor(random_state=42)

models["KNN"] = KNeighborsRegressor()
models["SVR"] = SVR()
models["MLP"] = MLPRegressor(max_iter=1000, random_state=42)

# Optional Models
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        verbosity=0
    )
except:
    pass

try:
    from catboost import CatBoostRegressor
    models["CatBoost"] = CatBoostRegressor(
        verbose=0,
        random_state=42
    )
except:
    pass

print("Total Models:", len(models))
print(models.keys())

### STEP 3 — Run All Models Automatically

In [ ]:
results = []

for name, model in models.items():
    
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Metrics
    r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({
        "Model": name,
        "R2": round(r2,4),
        "RMSE": round(rmse,4),
        "MAE": round(mae,4)
    })

print("Done")

### STEP 4 — Create Result Table

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R2", ascending=False)

display(results_df)

### STEP 5 — Visualize R² Scores

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x="R2", y="Model", data=results_df)
plt.title("Model Comparison by R² Score")
plt.show()

### Phase 5A - Cross Validation

### STEP 1 — Import CV

In [ ]:
from sklearn.model_selection import cross_val_score

### STEP 2 — Linear Regression CV

In [ ]:
from sklearn.linear_model import LinearRegression

pipe_lr = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])

scores_lr = cross_val_score(
    pipe_lr, X, y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Linear Regression CV Scores:", scores_lr)
print("Mean R²:", scores_lr.mean())
print("Std:", scores_lr.std())

### STEP 3 — Ridge CV

In [ ]:
from sklearn.linear_model import Ridge

pipe_ridge = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge())
])

scores_ridge = cross_val_score(
    pipe_ridge, X, y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Ridge CV Scores:", scores_ridge)
print("Mean R²:", scores_ridge.mean())
print("Std:", scores_ridge.std())

### STEP 4 — MLP CV

In [ ]:
from sklearn.neural_network import MLPRegressor

pipe_mlp = Pipeline([
    ("prep", preprocessor),
    ("model", MLPRegressor(max_iter=2000, random_state=42))
])

scores_mlp = cross_val_score(
    pipe_mlp, X, y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("MLP CV Scores:", scores_mlp)
print("Mean R²:", scores_mlp.mean())
print("Std:", scores_mlp.std())

### Why Cross-Validation R² is Lower Than Original Test R²?

The cross-validation R² score is typically **lower** than the original test R² score. This is expected and not a problem. Here's why:

| Aspect | Original Test R² | Cross-Validation R² |
|--------|-----------------|---------------------|
| **Data Used** | Only 80% training, 20% testing | Uses ALL data across 5 folds |
| **Evaluation** | Single train/test split | Average of 5 different splits |
| **Bias** | May be overly optimistic | More realistic, unbiased estimate |

**Key Reasons for Lower CV Score:**

1. **More Robust Evaluation**: CV tests the model on **every** data point, not just one test set. This gives a more honest assessment of generalization.

2. **No Data Leakage**: In CV, each fold is validated on data the model has never seen during training. The original test set might have been easier by chance.

3. **Variance Across Folds**: The std (standard deviation) shows how much performance varies - if std is high, the model is sensitive to the data split.

4. **Uses Full Dataset**: CV uses the entire dataset (X, y) for validation, while the original only uses a portion.

**Conclusion**: A lower CV R² is actually a more **reliable** measure of model performance. It tells you how well the model will generalize to new, unseen data.

In [ ]:
# STEP 1: Compare Train vs Test R² to Detect Overfitting
print("=" * 60)
print("STEP 1: Train vs Test R² Comparison (Overfitting Check)")
print("=" * 60)

train_test_comparison = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict on both train and test
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    
    # Calculate R² for both
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Gap indicates overfitting
    gap = train_r2 - test_r2
    
    train_test_comparison.append({
        "Model": name,
        "Train R²": round(train_r2, 4),
        "Test R²": round(test_r2, 4),
        "Gap": round(gap, 4),
        "Status": "⚠️ Overfit" if gap > 0.1 else "✅ OK"
    })

comparison_df = pd.DataFrame(train_test_comparison)
comparison_df = comparison_df.sort_values(by="Test R²", ascending=False)
display(comparison_df)

print("\n💡 If Gap > 0.1, the model is overfitting (memorizing training data)")

In [ ]:
# STEP 2: Run 5-Fold Cross-Validation on ALL Models
print("=" * 60)
print("STEP 2: 5-Fold Cross-Validation on All Models")
print("=" * 60)

cv_results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # 5-Fold CV
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="r2", n_jobs=-1)
    
    cv_results.append({
        "Model": name,
        "CV Mean R²": round(cv_scores.mean(), 4),
        "CV Std": round(cv_scores.std(), 4),
        "CV Min": round(cv_scores.min(), 4),
        "CV Max": round(cv_scores.max(), 4)
    })

cv_df = pd.DataFrame(cv_results)
cv_df = cv_df.sort_values(by="CV Mean R²", ascending=False)
display(cv_df)

print("\n💡 Lower std = more stable model across different data splits")

### Phase 5B - Comprehensive R² Score Validation

Let's test your R² score more thoroughly using multiple validation methods to ensure it's not inflated.</VSCode.Cell>
<VSCode.Cell id="new1" language="python">
# STEP 1: Compare Train vs Test R² to Detect Overfitting
print("=" * 60)
print("STEP 1: Train vs Test R² Comparison (Overfitting Check)")
print("=" * 60)

train_test_comparison = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict on both train and test
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    
    # Calculate R² for both
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Gap indicates overfitting
    gap = train_r2 - test_r2
    
    train_test_comparison.append({
        "Model": name,
        "Train R²": round(train_r2, 4),
        "Test R²": round(test_r2, 4),
        "Gap": round(gap, 4),
        "Status": "⚠️ Overfit" if gap > 0.1 else "✅ OK"
    })

comparison_df = pd.DataFrame(train_test_comparison)
comparison_df = comparison_df.sort_values(by="Test R²", ascending=False)
display(comparison_df)

print("\n💡 If Gap > 0.1, the model is overfitting (memorizing training data)")
</VSCode.Cell>
<VSCode.Cell id="new2" language="python">
# STEP 2: Run 5-Fold Cross-Validation on ALL Models
print("=" * 60)
print("STEP 2: 5-Fold Cross-Validation on All Models")
print("=" * 60)

cv_results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # 5-Fold CV
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="r2", n_jobs=-1)
    
    cv_results.append({
        "Model": name,
        "CV Mean R²": round(cv_scores.mean(), 4),
        "CV Std": round(cv_scores.std(), 4),
        "CV Min": round(cv_scores.min(), 4),
        "CV Max": round(cv_scores.max(), 4)
    })

cv_df = pd.DataFrame(cv_results)
cv_df = cv_df.sort_values(by="CV Mean R²", ascending=False)
display(cv_df)

print("\n💡 Lower std = more stable model across different data splits")
</VSCode.Cell>
<VSCode.Cell id="new3" language="python">
# STEP 3: Run 10-Fold CV for More Robust Estimation
print("=" * 60)
print("STEP 3: 10-Fold Cross-Validation (More Robust)")
print("=" * 60)

cv10_results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # 10-Fold CV
    cv10_scores = cross_val_score(pipe, X, y, cv=10, scoring="r2", n_jobs=-1)
    
    cv10_results.append({
        "Model": name,
        "10-Fold CV Mean": round(cv10_scores.mean(), 4),
        "10-Fold CV Std": round(cv10_scores.std(), 4)
    })

cv10_df = pd.DataFrame(cv10_results)
cv10_df = cv10_df.sort_values(by="10-Fold CV Mean", ascending=False)
display(cv10_df)
</VSCode.Cell>
<VSCode.Cell id="new4" language="python">
# STEP 4: Compare Original Test R² vs CV R²
print("=" * 60)
print("STEP 4: Original Test R² vs CV R² Comparison")
print("=" * 60)

# Merge results
merged = results_df.merge(cv_df, on="Model")
merged = merged.rename(columns={"R2": "Original Test R²", "CV Mean R²": "5-Fold CV R²"})
merged["Difference"] = merged["Original Test R²"] - merged["5-Fold CV R²"]
merged = merged.sort_values(by="5-Fold CV R²", ascending=False)

display(merged[["Model", "Original Test R²", "5-Fold CV R²", "Difference"]])

print("\n💡 Large Difference (> 0.1) suggests original test score may be inflated")
</VSCode.Cell>
<VSCode.Cell id="new5" language="python">
# STEP 5: Leave-One-Out CV (Most Robust)
print("=" * 60)
print("STEP 5: Leave-One-Out CV (LOOCV) - Most Robust")
print("=" * 60)

from sklearn.model_selection import LeaveOneOut

# Test top 3 models with LOOCV
top_models = ["Random Forest", "Gradient Boosting", "XGBoost"]

for name in top_models:
    if name in models:
        pipe = Pipeline([
            ("prep", preprocessor),
            ("model", models[name])
        ])
        
        loo = LeaveOneOut()
        loo_scores = cross_val_score(pipe, X, y, cv=loo, scoring="r2", n_jobs=-1)
        
        print(f"\n{name}:")
        print(f"  LOOCV Mean R²: {loo_scores.mean():.4f}")
        print(f"  LOOCV Std: {loo_scores.std():.4f}")
        print(f"  LOOCV Min: {loo_scores.min():.4f}")
        print(f"  LOOCV Max: {loo_scores.max():.4f}")
</VSCode.Cell>
<VSCode.Cell id="new6" language="markdown">
### Summary: R² Score Validation Results

| Validation Method | Description | Reliability |
|------------------|-------------|-------------|
| **Original Test R²** | Single 80/20 split | ⚠️ May be inflated |
| **5-Fold CV R²** | Average of 5 splits | ✅ Good |
| **10-Fold CV R²** | Average of 10 splits | ✅✅ Better |
| **LOOCV R²** | Each sample as test | ✅✅✅ Most Robust |
| **Train-Test Gap** | Detects overfitting | ✅ Important |

**Key Findings:**
- If Train R² >> Test R² (> 0.1 gap) → Overfitting
- If CV R² much lower than Test R² → Original score may be inflated
- Low CV std → Model is stable across different data splits

In [ ]:
df.info()

### PHASE 5C — Tune Ridge (Most Important)

### Tune Alpha

In [ ]:
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 5, 10, 20, 50]
}

grid = GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Alpha:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

### Evaluate Tuned Ridge

In [ ]:
best_ridge = grid.best_estimator_

y_pred = best_ridge.predict(X_test)

print("Tuned Ridge R²:", r2_score(y_test, y_pred))

### PHASE 5D — Tune MLP (Can Beat Linear)

In [ ]:
mlp_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", MLPRegressor(random_state=42, max_iter=3000))
])

param_grid = {
    "model__hidden_layer_sizes": [(50,), (100,), (100,50)],
    "model__alpha": [0.0001, 0.001, 0.01],
    "model__learning_rate_init": [0.001, 0.01]
}

grid_mlp = GridSearchCV(
    mlp_pipe,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_mlp.fit(X_train, y_train)

print("Best MLP Params:", grid_mlp.best_params_)
print("Best CV Score:", grid_mlp.best_score_)

### PHASE 6 — Feature Engineering + Final Highest R²

### STEP 1 — Create Copy

In [ ]:
df_fe = df.copy()

### STEP 2 — New Features

In [ ]:
# Interaction Features
df_fe["distance_prep"] = df_fe["Distance_km"] * df_fe["Preparation_Time_min"]

# Avoid divide by zero
df_fe["exp_distance"] = df_fe["Courier_Experience_yrs"] / (df_fe["Distance_km"] + 1)

# Rush hour flag
df_fe["rush_hour"] = df_fe["Time_of_Day"].apply(
    lambda x: 1 if x in ["Morning", "Evening"] else 0
)

# Bad weather flag
df_fe["bad_weather"] = df_fe["Weather"].apply(
    lambda x: 1 if x in ["Rainy", "Snowy", "Foggy"] else 0
)

# High traffic flag
df_fe["traffic_high"] = df_fe["Traffic_Level"].apply(
    lambda x: 1 if x == "High" else 0
)

df_fe.head()

### STEP 3 — Prepare New X and y

In [ ]:
X = df_fe.drop("Delivery_Time_min", axis=1)
y = df_fe["Delivery_Time_min"]

### STEP 4 — Update Column Lists

In [ ]:
cat_cols = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

num_cols = [
    "Distance_km",
    "Preparation_Time_min",
    "Courier_Experience_yrs",
    "distance_prep",
    "exp_distance",
    "rush_hour",
    "bad_weather",
    "traffic_high"
]

### STEP 5 — Rebuild Preprocessor

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", MinMaxScaler(), num_cols)
])

### STEP 6 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

### STEP 7 — Test Ridge + Linear Again

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=0.1)
}

for name, model in models.items():
    
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    
    print(name, "R² =", round(r2_score(y_test, pred),4))

### New CV Test On "df_fe copy"

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge(alpha=0.1))
])

scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("CV Scores:", scores)
print("Mean R²:", scores.mean())
print("Std:", scores.std())

### Correct Cross Validation Code for All Winners

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
import pandas as pd

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "MLP": MLPRegressor(
        hidden_layer_sizes=(100,),
        alpha=0.001,
        learning_rate_init=0.01,
        max_iter=3000,
        random_state=42
    )
}

cv_results = []

for name, model in models.items():
    
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )
    
    cv_results.append({
        "Model": name,
        "Mean_R2": scores.mean(),
        "Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values("Mean_R2", ascending=False)
print(cv_df)

### PHASE 7 — Final Highest Model

### STEP 1 — Imports

In [ ]:
from sklearn.ensemble import StackingRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

### STEP 2 — Build Stacking Model

In [ ]:
base_models = [
    ("ridge", Ridge(alpha=0.1)),
    ("gbr", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )),
    ("et", ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ))
]

stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(),
    n_jobs=-1,
    passthrough=True
)

### STEP 3 — Full Pipeline + Log Target

In [ ]:
final_model = Pipeline([
    ("prep", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=stack_model,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

### STEP 4 — Train Final Model

In [77]:
final_model.fit(X_train, y_train)

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### STEP 5 — Predict

In [ ]:
y_pred = final_model.predict(X_test)

### STEP 6 — Evaluate

In [76]:
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)

print("Final Model Results")
print("R² :", round(r2,4))
print("RMSE:", round(rmse,4))
print("MAE :", round(mae,4))

Final Model Results
R² : 0.8246
RMSE: 8.8678
MAE : 5.8813


### STEP 7 — Actual vs Predicted Graph

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual Delivery Time")
plt.ylabel("Predicted Delivery Time")
plt.title("Actual vs Predicted")
plt.show()

### STEP 8 — Save Best Model

In [75]:
import joblib
joblib.dump(final_model, "best_robust_model_final.pkl")
print("Model Saved Successfully")

Model Saved Successfully
